<img align="right" src="https://github.com/codeBMB/April26_Rutgers-RCSB/raw/main/images/PyBMB_logo.png" width="150" height="150" />

# Project: CodeBMB: Computational Literacy for Biochemistry and Molecular Biology Education
## Notebook 005: Enzyme Kinetics Analysis — Faculty Template

**Purpose:**
This notebook guides students through a complete enzyme kinetics workflow
using fumarase as the example enzyme. Students calculate enzyme activity
from raw absorbance data, determine kinetic parameters (Km and Vmax) by
three methods — Michaelis-Menten curve fitting, Lineweaver-Burk plot, and
Eadie-Hofstee plot — and compare the results from each method.

Faculty can adapt this template to any enzyme by modifying the variables
at each 🔧 TEMPLATE MODIFICATION POINT.

In this lesson students will learn to:
- Convert raw spectrophotometric data to enzyme activity (U/mg)
- Plot and interpret a Michaelis-Menten curve
- Perform non-linear curve fitting using `scipy.optimize.curve_fit`
- Perform linear regression on linearized kinetic data
- Construct and interpret Lineweaver-Burk and Eadie-Hofstee plots
- Compare kinetic parameters derived by different methods

> 🔧 **This notebook is a faculty template.**
> Fumarase is used as the working example throughout.
> Every place where you adapt it to your own enzyme is marked with 🔧.

**Complete List of Template Modification Points:**

| Section | Variable | Current Value | What to Change |
|---|---|---|---|
| 3a | `enzyme_name` | `'Fumarase'` | Your enzyme name |
| 3a | `substrate_name` | `'Malate'` | Your substrate name |
| 3a | `product_name` | `'Fumarate'` | Your product name |
| 3a | `wavelength` | `250` | Your measurement wavelength (nm) |
| 3a | `molextinction` | `1.440` | Your molar extinction coefficient (mM⁻¹ cm⁻¹) |
| 3a | `volume` | `0.2` | Your reaction volume (mL) |
| 3a | `substrate_units` | `'mM'` | Your substrate concentration units |
| 3b | `raw_data` | Example values | Your measured ΔAU/min values |
| 3b | Substrate concentrations | `[0.0, 0.25, 1.0, 2.5, 10.0, 25.0]` | Your concentrations |
| 4 | `mg_of_enzyme` | `0.002` | Your mg of enzyme per reaction |
| 5.2 | `vmax_visual_estimate` | `25.0` | Your visual Vmax estimate |
| 5.2 | `km_visual_estimate` | `2.0` | Your visual Km estimate |
| 9 | Interpretation questions | Fumarase context | Your enzyme context |

**Input Data:**
* **Description:** Average ΔAU/min measured at each substrate concentration
  for fumarase activity assay, corrected to 1 cm path length.
  Enzyme activity is measured as fumarate production at 250 nm.
* **Source:** Student experimental data entered manually
* **Access:** Entered directly in Section 3b

**Libraries:**
* `numpy` — numerical arrays and mathematical operations
* `matplotlib` — plotting MM, LB, and EH graphs
* `scipy.optimize.curve_fit` — non-linear regression for MM fitting
* `scipy.stats.linregress` — linear regression for LB and EH plots
* `pandas` — comparison table of results from all three methods

**Status with Date:** April 2025 — Faculty template version

**License**

<img src="https://github.com/codeBMB/April26_Rutgers-RCSB/raw/main/images/by-nc-sa.png" width="100"/>
CC BY-NC-SA — reusers may distribute, remix, adapt, and build upon
the material for noncommercial purposes only, with attribution,
under identical terms.

---
**Authorship:** Wally Novak, Paul Craig

**Acknowledgements:** This workshop is supported by NSF IUSE 2518733.
Original notebook developed for Biochemistry at Wabash College.

**Contact Info:** codingBMB@gmail.com

# 0. How to Run This Notebook

To run a cell, click the ▶ play button on the left side of the cell.

![run button image](https://github.com/wallynovak/biochemistry_seq_analysis/blob/main/images/run.png?raw=1)

A cell is still running if you see a stop button with a moving circle.
A completed cell shows a number in brackets (e.g. [1]) and a checkmark.

**Please ensure every cell finishes before running the next one.**
**Cells must be run in order from top to bottom.**

> 🔧 **For faculty adapting this template:** Before sharing with
> students, complete all 🔧 sections, run all cells once to
> confirm everything works, then go to
> **Runtime → Restart and clear all outputs** before distributing.

# 1. Environment Setup and Libraries

## Libraries Used in This Notebook

| Library | Purpose |
|---|---|
| `numpy` | Numerical arrays and mathematical operations |
| `matplotlib` | Creating all plots in this notebook |
| `scipy.optimize` | Non-linear curve fitting for Michaelis-Menten |
| `scipy.stats` | Linear regression for Lineweaver-Burk and Eadie-Hofstee |
| `pandas` | Building the final comparison table |

Run this cell first — everything below depends on it.

In [ ]:
# Run this cell to import all required libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.stats import linregress
import pandas as pd

print("✅ All libraries imported successfully!")

# 2. Theory Block

> 🔧 **FACULTY NOTE — Theory Block**
> The explanation below uses fumarase and malate as the example.
> The equations and methods are universal — only the biological
> context needs updating for a different enzyme. Replace
> the highlighted sections with a description of your own
> enzyme and its physiological role.

---

## The Enzyme: Fumarase

**Fumarase** (fumarate hydratase, EC 4.2.1.2) catalyzes the reversible
hydration of fumarate to malate — a key step in the citric acid cycle:

$$Fumarate + H_2O \rightleftharpoons L-Malate$$

We measure the **reverse reaction** (malate → fumarate) by monitoring
the increase in absorbance at **250 nm**, where fumarate absorbs
but malate does not. This allows us to directly track product formation.

---

## Converting Absorbance to Enzyme Activity

The **Beer-Lambert Law** (introduced in Notebook 004) connects
absorbance to concentration:

$$A = \varepsilon \cdot l \cdot c$$

where $\varepsilon$ is the molar extinction coefficient,
$l$ is path length, and $c$ is concentration.

Rearranging to get the rate of product formation:

$$rate\ (\mu mol/min) = \frac{\Delta AU/min \times V_{rxn}}{\varepsilon}$$

where $V_{rxn}$ is the reaction volume in mL and $\varepsilon$ is in mM⁻¹ cm⁻¹.

One **Unit (U)** of enzyme activity = 1 µmol of product formed per minute.
Dividing by the mg of enzyme gives **specific activity in U/mg**.

---

## Michaelis-Menten Kinetics

The **Michaelis-Menten equation** describes how reaction velocity (v)
depends on substrate concentration ([S]):

$$v = \frac{V_{max} \cdot [S]}{K_m + [S]}$$

| Parameter | Symbol | Meaning |
|---|---|---|
| Maximum velocity | $V_{max}$ | Velocity when all enzyme active sites are saturated |
| Michaelis constant | $K_m$ | Substrate concentration at half $V_{max}$ |

**Biological interpretation:**
- A **low Km** means the enzyme has high affinity for its substrate
  (reaches half-saturation at low [S])
- A **high Vmax** means the enzyme turns over substrate rapidly
- $K_m$ is often used as an approximation of the dissociation constant
  of the enzyme-substrate complex

In **Notebook 002**, you used the MM equation directly to calculate
velocities from defined Km and Vmax values. Here we go the other
direction — we use experimental data to determine Km and Vmax.

---

## Three Methods for Determining Km and Vmax

We will use three approaches in this notebook:

| Method | Approach | Advantages | Disadvantages |
|---|---|---|---|
| **MM Curve Fitting** | Non-linear regression | Most accurate | Requires initial guesses |
| **Lineweaver-Burk** | Plot 1/v vs 1/[S] | Historically common; intuitive | Overemphasizes low-[S] points |
| **Eadie-Hofstee** | Plot v vs v/[S] | More uniform error distribution | Both axes contain v (not independent) |

---

## Lineweaver-Burk (Double Reciprocal) Plot

Taking the reciprocal of the Michaelis-Menten equation gives:

$$\frac{1}{v} = \frac{K_m}{V_{max}} \cdot \frac{1}{[S]} + \frac{1}{V_{max}}$$

This is a straight line (y = mx + b) where:
- **Slope** = $K_m / V_{max}$
- **Y-intercept** = $1 / V_{max}$
- **X-intercept** = $-1 / K_m$

---

## Eadie-Hofstee Plot

Rearranging the MM equation differently gives:

$$v = V_{max} - K_m \cdot \frac{v}{[S]}$$

This is a straight line (y = mx + b) where:
- **Y-axis** = v (reaction velocity)
- **X-axis** = v/[S]
- **Slope** = $-K_m$
- **Y-intercept** = $V_{max}$

# 3. Data Entry

## 3a. Define Your Enzyme and Assay Parameters

These variables are used throughout the notebook in calculations,
axis labels, and print statements. Change them once here and
the rest of the notebook updates automatically.

> 🔧 **TEMPLATE MODIFICATION POINT**
> Replace every value marked 🔧 with parameters for your enzyme.

In [ ]:
# ─────────────────────────────────────────────────────────────
# 🔧 TEMPLATE MODIFICATION POINT — Enzyme and assay parameters
# ─────────────────────────────────────────────────────────────

enzyme_name     = 'Fumarase'    # 🔧 Your enzyme name
substrate_name  = 'Malate'      # 🔧 Your substrate name
product_name    = 'Fumarate'    # 🔧 Your product name (what you measure)
wavelength      = 250           # 🔧 Detection wavelength (nm)
substrate_units = 'mM'          # 🔧 Substrate concentration units
activity_units  = 'U/mg'        # 🔧 Activity units (usually U/mg)

# Molar extinction coefficient for your product at your wavelength
# For fumarate at 250 nm: 1.440 mM-1 cm-1
# Find this value in the literature or your lab manual
molextinction = 1.440           # 🔧 mM-1 cm-1

# Volume of the reaction in mL
volume = 0.2                    # 🔧 mL

# Label of the no-enzyme or no-substrate control row
control_label = 'A'             # 🔧 Row ID for your blank/control

# ─────────────────────────────────────────────────────────────
# No changes needed below this line
# ─────────────────────────────────────────────────────────────
print(f"✅ Parameters set for: {enzyme_name}")
print(f"   Substrate:        {substrate_name} ({substrate_units})")
print(f"   Product measured: {product_name} at {wavelength} nm")
print(f"   ε:                {molextinction} mM⁻¹ cm⁻¹")
print(f"   Reaction volume:  {volume} mL")

## 3b. Enter Your Experimental Data

Enter your measured average ΔAU/min values at each substrate
concentration. The data is organized as a list of tuples:

~~~python
(row_label, average_deltaAU_per_min, substrate_concentration)
~~~

**Example values are pre-filled so the notebook runs immediately.**
Replace each value with your own experimental measurements.

> 🔧 **TEMPLATE MODIFICATION POINT — Your measured ΔAU/min values**
>
> For each row:
> - `row_label` — identifier for this concentration (e.g. "A", "B")
> - `average_deltaAU_per_min` — your measured rate at this concentration
>   (average of your replicates, already corrected to 1 cm path length)
> - `substrate_concentration` — the concentration in your substrate units
>
> The **control row** (label `'A'`, concentration `0.0`) represents
> your no-enzyme or no-substrate blank. Its activity will be subtracted
> as background from all other measurements.

**How to adjust the number of concentrations:**
- To **add** a concentration: add a new tuple to the list
- To **remove** a concentration: delete that tuple
- Make sure `control_label` in Section 3a matches the label you use
  for your blank row

In [ ]:
# ─────────────────────────────────────────────────────────────
# 🔧 TEMPLATE MODIFICATION POINT — Enter your data below
# Format: (row_label, average_deltaAU_per_min, substrate_concentration)
# Replace the example values with your own measured ΔAU/min
# ─────────────────────────────────────────────────────────────

raw_data = [
    ("A", 0.001,  0.0),    # 🔧 Control (blank) — no substrate or no enzyme
    ("B", 0.040,  0.25),   # 🔧 0.25 mM malate — replace with your value
    ("C", 0.120,  1.0),    # 🔧 1.0 mM malate — replace with your value
    ("D", 0.200,  2.5),    # 🔧 2.5 mM malate — replace with your value
    ("E", 0.300,  10.0),   # 🔧 10.0 mM malate — replace with your value
    ("F", 0.333,  25.0),   # 🔧 25.0 mM malate — replace with your value
]
# 🔧 Add or remove rows above to match your experimental design

# ─────────────────────────────────────────────────────────────
# Verify data entry — no changes needed below
# ─────────────────────────────────────────────────────────────
print(f"Data entered for {enzyme_name}:")
print(f"{'Row':<6} {'ΔAU/min':<12} {f'[{substrate_name}] ({substrate_units})':<20}")
print("-" * 42)
for row_id, rate, conc in raw_data:
    flag = " ← control" if row_id == control_label else ""
    print(f"{row_id:<6} {rate:<12.4f} {conc:<20}{flag}")

## 3c. Converting ΔAU/min to Enzyme Activity (µmol/min)

Using the Beer-Lambert Law (introduced in Notebook 004), we convert
raw absorbance rate to molar rate:

$$rate\ (\mu mol/min) = \frac{\Delta AU/min \times V_{rxn}}{\varepsilon}$$

The code below:
1. Finds the background activity from your control row
2. Subtracts the background from each measurement
3. Applies the equation above to get activity in µmol/min

> No modifications needed in this cell — it uses the parameters
> and data you defined in Sections 3a and 3b.

In [ ]:
# 3c. Calculate activity in µmol/min
# Uses parameters from Section 3a and data from Section 3b

# Step 1: Find the background (control) activity
background_rate = 0.0
for row_id, rate, conc in raw_data:
    if row_id == control_label:
        background_rate = rate
        break

print(f"Background rate (row {control_label}): {background_rate:.4f} ΔAU/min")
print()

# Step 2: Calculate activity for each row
activity = []   # will store (row_id, activity_umol_per_min, substrate_conc)

print(f"{'Row':<6} {'ΔAU/min (corrected)':<24} {'Activity (µmol/min)':<22} {f'[{substrate_name}] ({substrate_units})'}")
print("-" * 70)

for row_id, rate, conc in raw_data:
    # Subtract background
    corrected_rate = rate - background_rate
    if row_id == control_label:
        corrected_rate = 0.0   # control is the reference — set to zero

    # Apply Beer-Lambert conversion
    # activity (µmol/min) = ΔAU/min × V_rxn (mL) / ε (mM-1 cm-1)
    # Note: mM = µmol/mL, so units work out to µmol/min
    act = corrected_rate * volume / molextinction
    act = round(act, 6)
    activity.append((row_id, act, conc))
    print(f"{row_id:<6} {corrected_rate:<24.4f} {act:<22.6f} {conc}")

print()
print("✅ Activity calculation complete!")

# 4. Converting to Specific Activity (U/mg)

One **Unit (U)** of enzyme activity = 1 µmol product formed per minute.

Since we now have activity in µmol/min = U, we divide by the
mg of enzyme added to each reaction to get **specific activity** in U/mg:

$$Specific\ Activity\ (U/mg) = \frac{Activity\ (\mu mol/min)}{mg\ of\ enzyme}$$

> 🔧 **TEMPLATE MODIFICATION POINT**
> Set `mg_of_enzyme` to the mass of enzyme you added to each reaction.
>
> **How to calculate mg_of_enzyme:**
> If you added 2 µL of enzyme solution at a concentration of 0.5 mg/mL:
> mg_of_enzyme = 0.002 mL × 0.5 mg/mL = 0.001 mg
>
> Use the protein concentration you determined from your Bradford assay
> (Notebook 004!) to calculate this value.

In [ ]:
# ─────────────────────────────────────────────────────────────
# 🔧 TEMPLATE MODIFICATION POINT
# Set mg_of_enzyme to the mass of enzyme added to each reaction
# ─────────────────────────────────────────────────────────────

mg_of_enzyme = 0.002   # 🔧 mg of enzyme per reaction

# ─────────────────────────────────────────────────────────────
# No changes needed below
# ─────────────────────────────────────────────────────────────
activity_per_mg = []   # will store (row_id, U/mg, substrate_conc)

print(f"mg of enzyme per reaction: {mg_of_enzyme} mg")
print()
print(f"{'Row':<6} {f'[{substrate_name}] ({substrate_units})':<20} {f'Activity ({activity_units})':<20}")
print("-" * 50)

for row_id, act_umol_per_min, conc in activity:
    act_umg = round(act_umol_per_min / mg_of_enzyme, 4)
    activity_per_mg.append((row_id, act_umg, conc))
    print(f"{row_id:<6} {conc:<20} {act_umg:<20}")

# Store as separate lists for plotting
x_coords = [item[2] for item in activity_per_mg]   # substrate concentrations
y_coords = [item[1] for item in activity_per_mg]   # activities in U/mg

print()
print(f"✅ Specific activity calculated for all {len(activity_per_mg)} concentrations")

# 5. Michaelis-Menten Analysis

With our data in U/mg, we can now determine Km and Vmax.
We work through three steps:

1. **Plot the data** — visualize the MM curve shape
2. **Estimate Vmax and Km visually** — initial guesses for the fitting algorithm
3. **Fit the MM equation** — non-linear regression for accurate parameters

## 5.1 Plotting the Michaelis-Menten Curve

Before fitting any model, always look at your data first.
The MM curve should show:
- Near-zero activity at zero substrate (your control)
- Rising activity as substrate increases
- A plateau approaching Vmax at high substrate concentrations

> No modifications needed — axis labels update automatically
> from the parameters defined in Section 3a.

In [ ]:
# 5.1 Scatter plot of activity vs substrate concentration

plt.figure(figsize=(10, 6))
plt.scatter(x_coords, y_coords,
            color='purple',
            s=80,
            label=f'{enzyme_name} Activity')

plt.xlabel(f'[{substrate_name}] ({substrate_units})')
plt.ylabel(f'Activity ({activity_units})')
plt.title(f'{enzyme_name}: Activity vs. [{substrate_name}]')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print(f"Maximum observed activity: {max(y_coords):.4f} {activity_units}")
print(f"Maximum [{substrate_name}] tested: {max(x_coords):.1f} {substrate_units}")

## 5.2 Visual Estimation of Vmax and Km

Before running the curve-fitting algorithm, estimate Vmax and Km
from the plot above. These estimates serve as **initial guesses**
that help the fitting algorithm converge to the correct solution.

**Estimating Vmax:**
Look for the plateau of the curve. Vmax is the activity the curve
is approaching at saturating substrate concentrations.

**Estimating Km:**
Find the substrate concentration at half of your estimated Vmax.
That concentration is approximately Km.

> 🔧 **TEMPLATE MODIFICATION POINT**
> Update `vmax_visual_estimate` and `km_visual_estimate` based
> on what you see in the plot above.
> Include units in your thinking but enter numbers only.

In [ ]:
# ─────────────────────────────────────────────────────────────
# 🔧 TEMPLATE MODIFICATION POINT
# Update these estimates based on your plot above
# ─────────────────────────────────────────────────────────────

vmax_visual_estimate = 25.0   # 🔧 Your visual estimate of Vmax (U/mg)
km_visual_estimate   = 2.0    # 🔧 Your visual estimate of Km (mM)

# ─────────────────────────────────────────────────────────────
print(f"Your visual estimates for {enzyme_name}:")
print(f"  Vmax ≈ {vmax_visual_estimate} {activity_units}")
print(f"  Km   ≈ {km_visual_estimate} {substrate_units}")
print()
print("These will serve as initial guesses for the curve-fitting algorithm.")
print("The algorithm will refine these to find the best-fit values.")

## 5.3 Non-Linear Curve Fitting

We now fit the Michaelis-Menten equation to our data using
`scipy.optimize.curve_fit`. This function:

1. Takes our MM equation as a Python function
2. Takes our x and y data
3. Takes initial parameter guesses (`p0`)
4. Returns the best-fit values of Vmax and Km that minimize
   the sum of squared residuals

This is similar to `scipy.stats.linregress` from **Notebook 004**
but handles **non-linear** equations — the MM curve is not a straight line.

**About the control point:**
We exclude the control (0 mM substrate, 0 U/mg activity) from the
fitting. The MM equation is defined for S > 0, and including (0,0)
can introduce numerical instability.

> No modifications needed — uses your visual estimates as initial guesses.

In [ ]:
# 5.3 Non-linear curve fitting using the Michaelis-Menten equation

# Define the Michaelis-Menten equation as a Python function
def michaelis_menten(S, Vmax, Km):
    """
    Michaelis-Menten equation.
    S    = substrate concentration
    Vmax = maximum velocity
    Km   = Michaelis constant
    """
    return (Vmax * S) / (Km + S)

# Filter out the control point (0 concentration, 0 activity)
# to avoid numerical issues with the MM equation at S=0
fit_x = [x for i, x in enumerate(x_coords) if x != 0.0]
fit_y = [y for i, y in enumerate(y_coords) if x_coords[i] != 0.0]

print(f"Fitting MM equation to {len(fit_x)} data points (control excluded)")
print(f"Initial guesses: Vmax={vmax_visual_estimate}, Km={km_visual_estimate}")
print()

# Perform the curve fit
params, covariance = curve_fit(
    michaelis_menten,
    fit_x,
    fit_y,
    p0=[vmax_visual_estimate, km_visual_estimate]
)

# Extract fitted parameters
Vmax_fitted, Km_fitted = params

# Calculate R² for the non-linear fit
y_predicted = michaelis_menten(np.array(fit_x), Vmax_fitted, Km_fitted)
y_observed  = np.array(fit_y)
ss_res = np.sum((y_observed - y_predicted)**2)
ss_tot = np.sum((y_observed - np.mean(y_observed))**2)
R2_mm  = 1 - (ss_res / ss_tot)

print(f"--- {enzyme_name} Michaelis-Menten Fit ---")
print(f"Fitted Vmax: {Vmax_fitted:.4f} {activity_units}")
print(f"Fitted Km:   {Km_fitted:.4f} {substrate_units}")
print(f"R²:          {R2_mm:.4f}")
print()

# Generate smooth curve for plotting
S_fit = np.linspace(0, max(x_coords), 200)
V_fit = michaelis_menten(S_fit, Vmax_fitted, Km_fitted)

# Plot experimental data and fitted curve
plt.figure(figsize=(10, 6))
plt.scatter(x_coords, y_coords, color='purple', s=80,
            label='Experimental Data', zorder=5)
plt.plot(S_fit, V_fit, color='red', linestyle='--',
         label=f'MM Fit: Vmax={Vmax_fitted:.3f}, Km={Km_fitted:.3f}\nR²={R2_mm:.4f}')

# Mark Vmax and Km on the plot
plt.axhline(y=Vmax_fitted, color='gray', linestyle=':', alpha=0.7,
            label=f'Vmax = {Vmax_fitted:.3f} {activity_units}')
plt.axhline(y=Vmax_fitted/2, color='lightgray', linestyle=':', alpha=0.7,
            label=f'Vmax/2 = {Vmax_fitted/2:.3f}')
plt.axvline(x=Km_fitted, color='lightgray', linestyle=':', alpha=0.7,
            label=f'Km = {Km_fitted:.3f} {substrate_units}')

plt.xlabel(f'[{substrate_name}] ({substrate_units})')
plt.ylabel(f'Activity ({activity_units})')
plt.title(f'{enzyme_name}: Michaelis-Menten Curve Fitting')
plt.grid(True, alpha=0.3)
plt.legend(loc='lower right')
plt.show()

## 📝 Exercise 5.3 — Interpreting the Michaelis-Menten Fit

Answer the following in the code cell below:

1. Print the **difference** between your visual estimates and the
   fitted values for both Vmax and Km. How close were your estimates?

2. Is R² ≥ 0.99? Write an `if/elif/else` statement (from **Notebook 002**)
   that prints whether the fit is "Excellent", "Acceptable", or
   "Check your data" — the same pattern used in Notebook 004.

3. In plain language: what does the Km value tell you about
   how the enzyme interacts with its substrate?

In [ ]:
# 1. Difference between visual estimates and fitted values
print(f"Vmax estimate: {vmax_visual_estimate:.3f}  |  Vmax fitted: {Vmax_fitted:.4f}")
print(f"Km estimate:   {km_visual_estimate:.3f}  |  Km fitted:   {Km_fitted:.4f}")
# calculate differences here


# 2. Evaluate R² quality using if/elif/else


# 3. Write a plain-language interpretation of Km here as a comment or print statement


<details>
<summary>🔍 CLICK TO SEE ANSWER</summary>

~~~python
# 1. Differences
vmax_diff = abs(vmax_visual_estimate - Vmax_fitted)
km_diff   = abs(km_visual_estimate - Km_fitted)
print(f"Vmax difference: {vmax_diff:.4f} {activity_units}")
print(f"Km difference:   {km_diff:.4f} {substrate_units}")

# 2. Evaluate fit quality
if R2_mm >= 0.99:
    print("\nExcellent fit ✅")
elif R2_mm >= 0.95:
    print("\nAcceptable fit ⚠️")
else:
    print("\nCheck your data ❌")

# 3. Km interpretation
print(f"\nKm = {Km_fitted:.4f} mM means the enzyme reaches half its maximum")
print(f"velocity at {Km_fitted:.4f} mM {substrate_name.lower()}.")
print("A lower Km indicates higher substrate affinity.")
~~~

</details>

# 6. Lineweaver-Burk (Double Reciprocal) Plot

The **Lineweaver-Burk plot** linearizes the MM equation by taking
reciprocals of both sides:

$$\frac{1}{v} = \frac{K_m}{V_{max}} \cdot \frac{1}{[S]} + \frac{1}{V_{max}}$$

Plotting **1/v** (y-axis) vs **1/[S]** (x-axis) gives a straight line:

| Feature | Value | How to read it |
|---|---|---|
| **Slope** | $K_m / V_{max}$ | From linear regression |
| **Y-intercept** | $1 / V_{max}$ | Therefore $V_{max} = 1 / intercept$ |
| **X-intercept** | $-1 / K_m$ | Therefore $K_m = -1 / x\_intercept$ |

> ⚠️ **A note on the Lineweaver-Burk plot:**
> The LB plot overemphasizes data at **low substrate concentrations**
> because these points have the largest reciprocal values and
> therefore the greatest leverage on the regression line.
> Small errors in low-[S] measurements can significantly affect
> the fitted Km and Vmax. For this reason, the MM curve fit
> (Section 5.3) is generally considered more accurate.

> No modifications needed in the code below — axis labels update
> automatically from Section 3a parameters.

In [ ]:
# 6. Lineweaver-Burk plot and linear regression

# Filter out zero-activity and zero-concentration points
lb_x_raw = [x for i, x in enumerate(x_coords) if y_coords[i] != 0.0 and x != 0.0]
lb_y_raw = [y for i, y in enumerate(y_coords) if y != 0.0 and x_coords[i] != 0.0]

# Calculate reciprocals
reciprocal_S = np.array([1/s for s in lb_x_raw])
reciprocal_V = np.array([1/v for v in lb_y_raw])

# Linear regression on 1/[S] vs 1/v
slope_lb, intercept_lb, r_value_lb, p_value_lb, std_err_lb = linregress(
    reciprocal_S, reciprocal_V
)
R2_lb = r_value_lb**2

# Calculate Km and Vmax from LB parameters
Vmax_lb = 1 / intercept_lb
Km_lb   = slope_lb * Vmax_lb

print(f"--- {enzyme_name} Lineweaver-Burk Results ---")
print(f"Slope:      {slope_lb:.4f}")
print(f"Intercept:  {intercept_lb:.4f}")
print(f"R²:         {R2_lb:.4f}")
print(f"Vmax:       {Vmax_lb:.4f} {activity_units}")
print(f"Km:         {Km_lb:.4f} {substrate_units}")
print()

# Generate fit line extending into negative x territory
# to show x-intercept (-1/Km)
x_min_fit = min(reciprocal_S.min(), -1/Km_lb) * 1.3
x_max_fit = reciprocal_S.max() * 1.1
S_fit_lb  = np.linspace(x_min_fit, x_max_fit, 200)
V_fit_lb  = slope_lb * S_fit_lb + intercept_lb

# Create the plot
plt.figure(figsize=(10, 6))
plt.scatter(reciprocal_S, reciprocal_V, color='blue', s=80,
            label='Experimental Data', zorder=5)
plt.plot(S_fit_lb, V_fit_lb, color='green', linestyle='--',
         label=f'Linear Fit: y = {slope_lb:.4f}x + {intercept_lb:.4f}\nR² = {R2_lb:.4f}')

# Mark x-intercept (-1/Km) and y-intercept (1/Vmax)
plt.axhline(y=0, color='black', linewidth=0.8)
plt.axvline(x=0, color='black', linewidth=0.8)
plt.scatter([-1/Km_lb], [0], color='red', s=100, zorder=6,
            label=f'X-intercept = -1/Km = {-1/Km_lb:.4f}')
plt.scatter([0], [1/Vmax_lb], color='orange', s=100, zorder=6,
            label=f'Y-intercept = 1/Vmax = {1/Vmax_lb:.4f}')

plt.xlabel(f'1/[{substrate_name}] (1/{substrate_units})')
plt.ylabel(f'1/Activity (1/({activity_units}))')
plt.title(f'{enzyme_name}: Lineweaver-Burk Plot')
plt.grid(True, alpha=0.3)
plt.legend(loc='upper left')
plt.show()

## 📝 Exercise 6 — Calculating Km and Vmax from the LB Plot

The Lineweaver-Burk equation is:

$$\frac{1}{v} = \frac{K_m}{V_{max}} \cdot \frac{1}{[S]} + \frac{1}{V_{max}}$$

Using the `slope_lb` and `intercept_lb` values printed above:

1. Calculate **Vmax** from the y-intercept of the LB plot.
   *Hint: the y-intercept = 1/Vmax, so Vmax = 1/intercept*

2. Calculate **Km** from the slope and your Vmax.
   *Hint: slope = Km/Vmax, so Km = slope × Vmax*

3. Calculate **Km** alternatively from the x-intercept.
   *Hint: x-intercept = -1/Km, so Km = -1/x_intercept*
   *The x-intercept is where the line crosses zero on the y-axis:
   0 = slope × x + intercept → x = -intercept/slope*

4. Do both methods give the same Km? Should they?

In [ ]:
# Use slope_lb and intercept_lb from the regression above

# 1. Calculate Vmax from y-intercept
Vmax_from_intercept = # your code here

# 2. Calculate Km from slope and Vmax
Km_from_slope = # your code here

# 3. Calculate Km from x-intercept
x_intercept = # your code here (where does the line cross y=0?)
Km_from_x_intercept = # your code here

# 4. Print and compare
print(f"Vmax from y-intercept: ")
print(f"Km from slope:         ")
print(f"Km from x-intercept:   ")

<details>
<summary>🔍 CLICK TO SEE ANSWER</summary>

~~~python
# 1. Vmax from y-intercept
Vmax_from_intercept = 1 / intercept_lb
print(f"Vmax from y-intercept: {Vmax_from_intercept:.4f} {activity_units}")

# 2. Km from slope
Km_from_slope = slope_lb * Vmax_from_intercept
print(f"Km from slope:         {Km_from_slope:.4f} {substrate_units}")

# 3. Km from x-intercept
# At y=0: 0 = slope*x + intercept → x = -intercept/slope
x_intercept = -intercept_lb / slope_lb
Km_from_x_intercept = -1 / x_intercept
print(f"X-intercept:           {x_intercept:.4f}")
print(f"Km from x-intercept:   {Km_from_x_intercept:.4f} {substrate_units}")

# 4. Both methods should give the same answer since they are
# mathematically equivalent rearrangements of the same line equation
~~~

> **Why they should agree:** Both calculations come from the same
> linear regression line — they are just different ways of reading
> the same equation. Any difference is due to rounding.

</details>

# 7. Eadie-Hofstee Plot

The **Eadie-Hofstee plot** is another linearization of the MM equation.
Rearranging the MM equation gives:

$$v = V_{max} - K_m \cdot \frac{v}{[S]}$$

Plotting **v** (y-axis) vs **v/[S]** (x-axis) gives a straight line where:
- **Y-intercept** = $V_{max}$
- **Slope** = $-K_m$
- **X-intercept** = $V_{max} / K_m$

**Advantages over Lineweaver-Burk:**
The EH plot distributes experimental error more uniformly across
the data range, giving less weight to low-[S] measurements.
However, both axes (v and v/[S]) contain the measured variable v,
so errors in v affect both axes simultaneously.

> No modifications needed — axis labels update automatically.

In [ ]:
# 7. Eadie-Hofstee plot and linear regression

# Filter out zero-concentration and zero-activity points
eh_S = [x for i, x in enumerate(x_coords) if y_coords[i] != 0.0 and x != 0.0]
eh_V = [y for i, y in enumerate(y_coords) if y != 0.0 and x_coords[i] != 0.0]

# Calculate v/[S] for the x-axis
v_over_S = np.array([v / s for v, s in zip(eh_V, eh_S)])
v_array  = np.array(eh_V)

# Linear regression: v = Vmax - Km * (v/[S])
# Y = intercept + slope * X → slope = -Km, intercept = Vmax
slope_eh, intercept_eh, r_value_eh, p_value_eh, std_err_eh = linregress(
    v_over_S, v_array
)
R2_eh   = r_value_eh**2
Vmax_eh = intercept_eh
Km_eh   = -slope_eh

print(f"--- {enzyme_name} Eadie-Hofstee Results ---")
print(f"Slope:      {slope_eh:.4f}   (= -Km)")
print(f"Intercept:  {intercept_eh:.4f}   (= Vmax)")
print(f"R²:         {R2_eh:.4f}")
print(f"Vmax:       {Vmax_eh:.4f} {activity_units}")
print(f"Km:         {Km_eh:.4f} {substrate_units}")
print()

# Generate fit line from 0 to x-intercept
x_intercept_eh = Vmax_eh / Km_eh if Km_eh != 0 else np.max(v_over_S) * 1.5
x_max_plot = max(np.max(v_over_S), x_intercept_eh) * 1.1
X_fit_eh = np.linspace(0, x_max_plot, 200)
Y_fit_eh = slope_eh * X_fit_eh + intercept_eh

# Create the plot
plt.figure(figsize=(10, 6))
plt.scatter(v_over_S, v_array, color='darkorange', s=80,
            label='Experimental Data', zorder=5)
plt.plot(X_fit_eh, Y_fit_eh, color='blue', linestyle='--',
         label=f'v = {intercept_eh:.4f} + ({slope_eh:.4f})·(v/[S])\nR² = {R2_eh:.4f}')

# Mark y-intercept (Vmax) and x-intercept (Vmax/Km)
plt.scatter([0], [Vmax_eh], color='red', s=100, zorder=6,
            label=f'Y-intercept = Vmax = {Vmax_eh:.4f}')
plt.scatter([x_intercept_eh], [0], color='purple', s=100, zorder=6,
            label=f'X-intercept = Vmax/Km = {x_intercept_eh:.4f}')

plt.axhline(y=0, color='black', linewidth=0.8)
plt.axvline(x=0, color='black', linewidth=0.8)
plt.ylim(ymin=0)
plt.xlim(xmin=0)
plt.xlabel(f'v/[{substrate_name}] ({activity_units}/{substrate_units})')
plt.ylabel(f'v ({activity_units})')
plt.title(f'{enzyme_name}: Eadie-Hofstee Plot')
plt.grid(True, alpha=0.3)
plt.legend(loc='upper right')
plt.show()

## 📝 Exercise 7 — Calculating Km and Vmax from the EH Plot

The Eadie-Hofstee equation is:

$$v = V_{max} - K_m \cdot \frac{v}{[S]}$$

Using `slope_eh` and `intercept_eh` from the regression above:

1. Calculate **Vmax** from the y-intercept.
   *Hint: y-intercept = Vmax*

2. Calculate **Km** from the slope.
   *Hint: slope = -Km, so Km = -slope*

3. Calculate **Km** alternatively from the x-intercept.
   *Hint: at v=0: 0 = Vmax - Km × (v/[S]) → v/[S] = Vmax/Km
   So x_intercept = Vmax/Km → Km = Vmax/x_intercept*

4. Compare your EH values to the LB values from Exercise 6.
   Which gives a higher Km? Which gives a higher Vmax?

In [ ]:
# Use slope_eh and intercept_eh from the EH regression above

# 1. Vmax from y-intercept
Vmax_from_eh_intercept = # your code here

# 2. Km from slope
Km_from_eh_slope = # your code here

# 3. Km from x-intercept
# At v=0: 0 = Vmax - Km * (v/[S]) → v/[S] = Vmax/Km
eh_x_intercept = # your code here
Km_from_eh_x   = # your code here

# 4. Print your results
print(f"EH Vmax:           ")
print(f"EH Km (slope):     ")
print(f"EH Km (x-int):     ")
print()
print(f"LB Vmax:           {Vmax_lb:.4f}")
print(f"LB Km:             {Km_lb:.4f}")

<details>
<summary>🔍 CLICK TO SEE ANSWER</summary>

~~~python
# 1. Vmax from y-intercept
Vmax_from_eh_intercept = intercept_eh
print(f"EH Vmax:           {Vmax_from_eh_intercept:.4f} {activity_units}")

# 2. Km from slope
Km_from_eh_slope = -slope_eh
print(f"EH Km (slope):     {Km_from_eh_slope:.4f} {substrate_units}")

# 3. Km from x-intercept
eh_x_intercept = Vmax_from_eh_intercept / Km_from_eh_slope
Km_from_eh_x   = Vmax_from_eh_intercept / eh_x_intercept
print(f"EH x-intercept:    {eh_x_intercept:.4f}")
print(f"EH Km (x-int):     {Km_from_eh_x:.4f} {substrate_units}")

print(f"\nLB Vmax:           {Vmax_lb:.4f}")
print(f"LB Km:             {Km_lb:.4f}")
~~~

> **Comparing LB and EH:** The two plots often give slightly different
> Km and Vmax values because they weight the data differently.
> The LB plot gives more weight to low-[S] measurements;
> the EH plot distributes error more evenly.
> Neither is perfectly accurate — the non-linear MM fit (Section 5.3)
> is the most statistically appropriate method.

</details>

# 8. Comparison of All Three Methods

Now let's compare the Km and Vmax values determined by each method.
We use a pandas DataFrame — the same tool from **Notebook 003** —
to organize the results into a clean comparison table.

In [ ]:
# 8. Build a comparison table using pandas

comparison_df = pd.DataFrame({
    'Method':          ['Michaelis-Menten Fit',
                        'Lineweaver-Burk',
                        'Eadie-Hofstee'],
    f'Vmax ({activity_units})': [round(Vmax_fitted, 4),
                                  round(Vmax_lb, 4),
                                  round(Vmax_eh, 4)],
    f'Km ({substrate_units})':  [round(Km_fitted, 4),
                                  round(Km_lb, 4),
                                  round(Km_eh, 4)],
    'R²':              [round(R2_mm, 4),
                        round(R2_lb, 4),
                        round(R2_eh, 4)]
})

print(f"{enzyme_name} Kinetic Parameters — Method Comparison")
print("=" * 65)
print(comparison_df.to_string(index=False))
print()

# Calculate range of Vmax and Km across methods
vmax_values = [Vmax_fitted, Vmax_lb, Vmax_eh]
km_values   = [Km_fitted, Km_lb, Km_eh]

print(f"Vmax range: {min(vmax_values):.4f} – {max(vmax_values):.4f} {activity_units}")
print(f"Km range:   {min(km_values):.4f} – {max(km_values):.4f} {substrate_units}")
print()
print(f"Do all three methods agree closely on Vmax? {max(vmax_values)-min(vmax_values) < 0.1*max(vmax_values)}")
print(f"Do all three methods agree closely on Km?   {max(km_values)-min(km_values) < 0.1*max(km_values)}")

# 9. Results and Interpretation

Review your comparison table from Section 8 and consider:

**About the kinetic parameters:**
- Do all three methods give similar Km and Vmax values?
  If not, which method do you trust most and why?
- What does your Km value tell you about fumarase's affinity
  for malate? Is this a high-affinity or low-affinity interaction?
- At physiological concentrations of malate in the cell (~0.2–0.5 mM),
  is fumarase operating near its maximum velocity?

**About the methods:**
- Which plot had the highest R²? Does a higher R² necessarily
  mean that method gives more accurate kinetic parameters?
- Why does the LB plot tend to give more variable results than
  the EH plot or MM curve fitting?

**About experimental quality:**
- Were your replicates consistent? How confident are you in your
  ΔAU/min measurements?
- Are there any data points that look like outliers on your plots?

> 🔧 **TEMPLATE MODIFICATION POINT — Interpretation Questions**
> Replace the questions above with questions relevant to your own
> enzyme. Consider:
> - The physiological role of your enzyme
> - Expected Km values from the literature
> - How your results compare to published values

# 10. Wrap-Up

Congratulations on completing Notebook 005T!

In this notebook you:
- ✅ Converted raw ΔAU/min measurements to specific activity (U/mg)
  using Beer-Lambert Law from **Notebook 004**
- ✅ Plotted and visually estimated Km and Vmax
- ✅ Performed non-linear curve fitting using `scipy.optimize.curve_fit`
- ✅ Constructed and interpreted a Lineweaver-Burk plot
- ✅ Constructed and interpreted an Eadie-Hofstee plot
- ✅ Compared kinetic parameters from all three methods
  using a pandas DataFrame from **Notebook 003**

**Connections across the workshop:**

| Notebook | Key concept | How it appeared in 005T |
|---|---|---|
| 002 | Variables, loops, if statements | Data processing loops; if/elif/else for quality checks |
| 003 | DataFrames | Comparison table in Section 8 |
| 004 | Beer-Lambert Law, linear regression | Activity calculation; LB and EH linear fits |
| 005T | Non-linear regression | MM curve fitting — extending 004's linregress |

**The workflow is a reusable template.**
To adapt it to a different enzyme, change every 🔧 point:

| Section | What to change |
|---|---|
| 3a | Enzyme name, substrate, product, wavelength, ε, volume |
| 3b | Your measured ΔAU/min values and substrate concentrations |
| 4 | `mg_of_enzyme` — your protein amount per reaction |
| 5.2 | Visual estimates of Vmax and Km |
| 9 | Interpretation questions for your enzyme |

# Optional Challenge Questions

| Challenge | Topic | Difficulty |
|---|---|---|
| C1 | Overlay all three fitted curves on one plot | ⭐⭐ |
| C2 | Determine the effect of a competitive inhibitor | ⭐⭐⭐ |
| C3 | Bootstrap uncertainty estimation for Km and Vmax | ⭐⭐⭐⭐ |

---

## Challenge 1 — Overlay All Three Fits ⭐⭐

Create a single plot that shows:
1. Your experimental data points (scatter)
2. The MM curve fit (from Section 5.3)
3. The Lineweaver-Burk prediction converted back to v vs [S]
4. The Eadie-Hofstee prediction converted back to v vs [S]

*Hint: for LB, use `V = michaelis_menten(S, Vmax_lb, Km_lb)`*
*For EH, use `V = michaelis_menten(S, Vmax_eh, Km_eh)`*

Do the three curves agree closely? At which substrate concentrations
do they diverge most?

---

## Challenge 2 — Competitive Inhibition ⭐⭐⭐

In the presence of a **competitive inhibitor**, the apparent Km
increases but Vmax remains unchanged:

$$v = \frac{V_{max} \cdot [S]}{\alpha K_m + [S]}, \quad \alpha = 1 + \frac{[I]}{K_i}$$

Using the same experimental setup, collect data at two inhibitor
concentrations ([I] = 0, and [I] = 1 mM of a competitive inhibitor).

1. Fit the MM equation to each dataset
2. Compare Vmax and apparent Km
3. Calculate the inhibition constant Ki from:
   $K_i = [I] / (\alpha - 1)$ where $\alpha = K_{m,app} / K_m$

---

## Challenge 3 — Bootstrap Uncertainty Estimation ⭐⭐⭐⭐

`curve_fit` returns a covariance matrix that gives uncertainty
in the fitted parameters. Use bootstrapping to estimate 95%
confidence intervals for Km and Vmax:

1. Resample your data with replacement 1000 times
2. Fit the MM equation to each resampled dataset
3. Collect all 1000 Vmax and Km values
4. Calculate the 2.5th and 97.5th percentiles as confidence intervals
5. Plot histograms of the bootstrap distributions

~~~python
import random
bootstrap_vmaxes = []
bootstrap_kms    = []

for _ in range(1000):
    # Resample with replacement
    indices  = random.choices(range(len(fit_x)), k=len(fit_x))
    x_boot   = [fit_x[i] for i in indices]
    y_boot   = [fit_y[i] for i in indices]
    try:
        p, _ = curve_fit(michaelis_menten, x_boot, y_boot,
                         p0=[Vmax_fitted, Km_fitted])
        bootstrap_vmaxes.append(p[0])
        bootstrap_kms.append(p[1])
    except RuntimeError:
        pass  # skip failed fits

print(f"Vmax 95% CI: {np.percentile(bootstrap_vmaxes, 2.5):.4f} – {np.percentile(bootstrap_vmaxes, 97.5):.4f}")
print(f"Km 95% CI:   {np.percentile(bootstrap_kms, 2.5):.4f} – {np.percentile(bootstrap_kms, 97.5):.4f}")
~~~

## Notebook Sign-Off Checklist

**General:**
* [ ] **Purpose is clear** from the header without opening code cells
* [ ] **Every significant decision** has a markdown explanation
* [ ] **Kernel restarted** and all cells run top-to-bottom without error

**Template-specific (before sharing with students):**
* [ ] `enzyme_name`, `substrate_name`, `product_name` updated in Section 3a
* [ ] `wavelength` and `molextinction` match your assay in Section 3a
* [ ] `volume` matches your reaction volume in Section 3a
* [ ] All raw data values in Section 3b replaced with real measurements
* [ ] `mg_of_enzyme` in Section 4 set to your protein amount
* [ ] `vmax_visual_estimate` and `km_visual_estimate` updated in Section 5.2
* [ ] Interpretation questions in Section 9 updated for your enzyme
* [ ] All outputs cleared (Runtime → Restart and clear all outputs)
  before sharing with students